In [2]:
import pandas as pd

datos2023 = pd.read_csv('datos2023.csv')

In [3]:
datos2023['DIRECCION'] = datos2023['DIRECCION'].astype('string')

In [4]:
departamentos_unicos = datos2023['DEPARTAMENTO'].unique()
print(len(departamentos_unicos))
print(departamentos_unicos)

191


In [5]:
datos2023.info()

In [6]:
datos2023['CODIGO_DIVIPOLA'] = (
    datos2023['CODIGO_DEPARTAMENTO'].astype(str).str.zfill(2) +
    datos2023['CODIGO_MUNICIPIO'].astype(str).str.zfill(3)
).astype('int64')

In [7]:
datos2023.info()

In [8]:
dfcant2023 = datos2023['CANTIDAD'].value_counts()
print(dfcant2023)

CANTIDAD
1 3402532


In [9]:
dfmunicipios2023 = datos2023['CODIGO_DIVIPOLA'].unique().tolist()
print(dfmunicipios2023)
len(set(dfmunicipios2023))

1078

In [12]:
import pandas as pd

filepath = 'PPED-AreaMun-2018-2042VP (1).xlsx'

df = pd.read_excel(filepath, sheet_name='PobMunicipalxArea', skiprows=7)

df = df.dropna(subset=['ANO', 'AREA_GEOGRAFICA'])

df2023total = df[(df['ANO'] == 2023) & (df['AREA_GEOGRAFICA'] == 'Total')].copy()

dfresultado = df2023total[['DP', 'DPNOM', 'MPIO', 'DPMP', 'TOTAL']]

dfresultado.columns = ['CódigoDepartamento', 'Departamento', 'CODIGO_DIVIPOLA', 'MUNICIPIO', 'PoblacionTotal2023']

dfresultado = dfresultado.reset_index(drop=True)

outputpath = 'Poblacion2023TotalMunicipios.xlsx'
dfresultado.to_excel(outputpath, index=False)

print(f'Total de municipios extraidos: {len(dfresultado)}')

Total de municipios extraidos: 1123


In [13]:
poblacion2023 = dfresultado
poblacion2023.head()

In [15]:
dfmerge = poblacion2023[['CODIGO_DIVIPOLA', 'Departamento', 'MUNICIPIO', 'PoblacionTotal2023']].copy()

dfmerge['CODIGO_DIVIPOLA'] = pd.to_numeric(dfmerge['CODIGO_DIVIPOLA'], errors='coerce').astype('Int64')
datos2023['CODIGO_DIVIPOLA'] = pd.to_numeric(datos2023['CODIGO_DIVIPOLA'], errors='coerce').astype('Int64')

df2023total = datos2023.merge(
    dfmerge,
    on='CODIGO_DIVIPOLA',
    how='left',
    suffixes=('', '_pob')
)

df2023total['DEPARTAMENTO'] = df2023total['Departamento'].fillna(df2023total['DEPARTAMENTO'])
df2023total['MUNICIPIO'] = df2023total['MUNICIPIO_pob'].fillna(df2023total['MUNICIPIO'])

df2023total = df2023total.rename(columns={'PoblacionTotal2023': 'PoblacionTotalMunicipio2023'})
df2023total = df2023total.drop(columns=['Departamento', 'MUNICIPIO_pob'])

In [16]:
df2023total.info()

In [17]:
departamentos_unicos = df2023total['DEPARTAMENTO'].unique()
print(len(departamentos_unicos))
print(departamentos_unicos)

33


In [19]:
municipio2023 = (
    df2023total
    .groupby(['CODIGO_DIVIPOLA', 'DEPARTAMENTO', 'MUNICIPIO'], as_index=False)
    .agg(
        comparendos=('CANTIDAD', 'sum'),
        poblacion=('PoblacionTotalMunicipio2023', 'max')
    )
)

municipio2023['poblacion'] = municipio2023['poblacion'].replace(0, pd.NA)

municipio2023['tasacomparendos100k'] = (
    municipio2023['comparendos'] / municipio2023['poblacion'] * 100000
)

totalcomparendos_calc = municipio2023['comparendos'].sum()
totalcomparendos2023 = datos2023['CANTIDAD'].sum()

print(f'Total comparendos calculado desde agrupacion municipal: {totalcomparendos_calc}')
print(f'totalcomparendos2023: {totalcomparendos2023}')
print(f'Coincide: {totalcomparendos_calc == totalcomparendos2023}')
print(municipio2023.sort_values('tasacomparendos100k', ascending=False).head(10))

Total comparendos calculado desde agrupacion municipal: 3403010
totalcomparendos2023: 3403010
Coincide: True


In [50]:
df2023total.to_csv('df2023total.csv', index=False)